# Nền tảng 3 — Đếm tham số, FLOPs và thiết kế công bằng

Notebook này trả lời ba câu hỏi mà báo cáo bắt buộc phải trả lời đúng:

1. **Model d6/d8/d10 có bao nhiêu tham số, và con số nào mới là con số nên báo cáo?**
2. **Một lượt train tốn bao nhiêu phép tính, và có bao nhiêu phần trăm năng lực GPU thực sự được dùng?**
3. **Khi hai tokenizer nén văn bản khác nhau, "cho hai model học bằng nhau" nghĩa là gì?** Đây là chỗ thiết kế
   của project có thể sai một cách âm thầm, nên mục 6–8 dành riêng cho nó.

Mọi con số tính trực tiếp từ mã nguồn nanochat trong `third_party/` và từ file kết quả trong `kaggle/outputs/`.

**Chạy bằng kernel pixi của project.**

In [ ]:
import contextlib
import io
import json
import math
import sys
from pathlib import Path

import numpy as np
import torch

ROOT = Path.cwd()
while not (ROOT / "pixi.toml").exists() and ROOT != ROOT.parent:   # tìm gốc repo, chạy được từ mọi thư mục
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "third_party" / "nanochat"))
RUNS = ROOT / "kaggle" / "outputs"

from nanochat.gpt import GPT, GPTConfig, has_ve
from vitok.conditions import BASE_SEQ, SEQS_PER_STEP, load_cpt, seq_len, train_args

cpt = load_cpt(RUNS / "vitok-data" / "compression-16k.json")   # chars/token của từng điều kiện
VOCAB = 16009          # 16.000 token học được + 9 special token của nanochat
PAD_TO = 64
print("repo:", ROOT)
print("torch:", torch.__version__)

## 1. Hình dạng của model: mọi thứ suy ra từ `depth`

nanochat chọn siêu tham số theo một trục duy nhất là **độ sâu** $L$ (`--depth`), phần còn lại suy ra:

| Đại lượng | Ký hiệu | Quy tắc trong nanochat | d6 | d8 | d10 |
|---|---|---|---|---|---|
| Số lớp | $L$ | tham số đầu vào | 6 | 8 | 10 |
| Chiều residual | $d$ | $d = 64L$ (`aspect_ratio = 64`) | 384 | 512 | 640 |
| Chiều mỗi đầu | $d_h$ | cố định 128 (`head_dim`) | 128 | 128 | 128 |
| Số đầu | $h$ | $h = d/d_h$ | 3 | 4 | 5 |

Quy tắc $d \propto L$ gọi là giữ **tỷ lệ khung** (aspect ratio) không đổi: model sâu hơn thì cũng rộng ra theo.
Đây là một **quy ước đơn giản hoá** để có một núm vặn duy nhất, không phải một định luật: Kaplan (2020) cho thấy
trong một khoảng tỷ lệ khung khá rộng, loss gần như chỉ phụ thuộc tổng số tham số chứ ít phụ thuộc hình dạng. Hệ quả cần nhớ cho mục 2: số tham số của
phần thân tăng theo $d^2 L \propto L^3$, nên **đi từ d6 lên d10 không phải tăng 1,67 lần mà khoảng 4,6 lần**.

## 2. Đếm tham số bằng tay

Liệt kê từng ma trận trong một lớp, với $h\,d_h = d$ (số đầu nhân chiều mỗi đầu đúng bằng chiều residual) và
$h_{kv} = h$ trong cấu hình project (không dùng GQA thu gọn):

| Ma trận | Kích thước | Số tham số |
|---|---|---|
| `c_q` (query) | $d \times h d_h$ | $d^2$ |
| `c_k` (key) | $d \times h_{kv} d_h$ | $d^2$ |
| `c_v` (value) | $d \times h_{kv} d_h$ | $d^2$ |
| `c_proj` (đầu ra attention) | $h d_h \times d$ | $d^2$ |
| `c_fc` (MLP vào) | $d \times 4d$ | $4d^2$ |
| `c_proj` (MLP ra) | $4d \times d$ | $4d^2$ |

Cộng lại: $4d^2 + 8d^2 = 12d^2$ mỗi lớp, nên **phần thân** (non-embedding) có

$$N_{\text{thân}} = 12\,d^2 L$$

Đây là công thức được dùng ở mọi nơi trong tài liệu scaling law, và là con số project báo cáo khi nói "d8 có 25M
tham số".

Ngoài phần thân còn ba bảng tra cứu, mỗi bảng có hàng bằng kích thước vocab:

| Thành phần | Kích thước | Ghi chú |
|---|---|---|
| `wte` (embedding đầu vào) | $V_{\text{pad}} \times d$ | tra cứu, không phải matmul |
| `lm_head` (unembedding) | $d \times V_{\text{pad}}$ | có matmul, tốn FLOPs |
| `value_embeds` | $n_{ve} \times V_{\text{pad}} \times d$ | ResFormer-style, chỉ ở **một nửa số lớp** |

Quy tắc chọn lớp có value embedding, đọc thẳng từ `nanochat/gpt.py`:

```python
def has_ve(layer_idx, n_layer):
    return layer_idx % 2 == (n_layer - 1) % 2
```

tức các lớp **cùng tính chẵn lẻ với lớp cuối**. Với $L$ chẵn thì đó là các lớp lẻ, và $n_{ve} = L/2$. Đây chính
là chỗ tôi đếm sai hai lần trước khi đọc kỹ hàm này: d10 có **5** bảng value embedding chứ không phải 4.

Cell dưới cộng tay từng thành phần rồi đối chiếu với model nanochat dựng trên `meta device` — thiết bị giả, chỉ
cấp phát hình dạng chứ không cấp phát bộ nhớ thật, nên dựng được model 121M tham số trên máy 6GB trong tích tắc.

In [ ]:
def dem_tay(depth, vocab=VOCAB, pad_to=PAD_TO):
    d = 64 * depth
    v = ((vocab + pad_to - 1) // pad_to) * pad_to        # đệm lên bội số 64
    n_head = d // 128
    n_ve = sum(1 for i in range(depth) if has_ve(i, depth))

    than = 12 * d * d * depth                            # 4d² attention + 8d² MLP, mỗi lớp
    wte = v * d
    lm_head = v * d
    value_embeds = n_ve * v * d
    # các tham số lặt vặt: scalar mỗi lớp + smear + backout + cổng cho value embedding
    vun_vat = 2 * depth + 24 + 1 + 1 + n_ve * 12 * n_head
    return {"d": d, "v_pad": v, "n_head": n_head, "n_ve": n_ve, "than": than, "wte": wte,
            "lm_head": lm_head, "value_embeds": value_embeds, "vun_vat": vun_vat,
            "tong": than + wte + lm_head + value_embeds + vun_vat}


def dung_model(depth, vocab=VOCAB, seq_len=1024):
    d = 64 * depth
    cfg = GPTConfig(sequence_len=seq_len, vocab_size=vocab, n_layer=depth,
                    n_head=d // 128, n_kv_head=d // 128, n_embd=d, window_pattern="L")
    with torch.device("meta"), contextlib.redirect_stdout(io.StringIO()):   # nuốt log "Padding vocab_size"
        return GPT(cfg)


for depth in (6, 8, 10):
    tay = dem_tay(depth)
    m = dung_model(depth)
    that = sum(p.numel() for p in m.parameters())
    assert tay["tong"] == that, (depth, tay["tong"], that)
    print(f"d{depth}: đếm tay = {tay['tong']:,} = model thật = {that:,}  ✓")

Bảng phân rã đầy đủ. Cột cuối là thứ đáng chú ý nhất: **ở d6, gần 3/4 tham số nằm trong các bảng tra cứu**, chỉ
1/4 là phần thân thực sự làm việc tính toán. Đó là lý do câu "model d6 có 41 triệu tham số" gây hiểu nhầm về
năng lực của nó, và là lý do mọi bảng trong báo cáo dùng cột `non-embedding params`.

In [ ]:
print(" model | d   | n_ve | thân (12d²L)  | wte+lm_head+ve | tổng        | tỷ lệ tra cứu")
for depth in (6, 8, 10):
    t = dem_tay(depth)
    tra_cuu = t["wte"] + t["lm_head"] + t["value_embeds"]
    print(f" d{depth:<4} | {t['d']:<3} | {t['n_ve']:^4} | {t['than']:>13,} | {tra_cuu:>14,} |"
          f" {t['tong']:>11,} | {tra_cuu / t['tong']:>6.1%}")

print("\ntăng từ d6 lên d10:")
a, b = dem_tay(6), dem_tay(10)
print(f"  số lớp    : {10 / 6:.2f}×")
print(f"  phần thân : {b['than'] / a['than']:.2f}×   (vì 12d²L ∝ L³)")
print(f"  tổng      : {b['tong'] / a['tong']:.2f}×   (bị các bảng tra cứu kéo xuống)")

print("\nnếu dùng vocab 32k thay vì 16k:")
for depth in (6, 8, 10):
    t16, t32 = dem_tay(depth, 16009), dem_tay(depth, 32009)
    print(f"  d{depth}: {t16['tong']:,} -> {t32['tong']:,}  (+{t32['tong'] / t16['tong'] - 1:.0%}),"
          f" phần thân không đổi: {t32['than'] == t16['than']}")

## 3. Đệm vocab lên bội số 64

Trong `GPT.__init__`:

```python
padded_vocab_size = ((config.vocab_size + pad_vocab_size_to - 1) // pad_vocab_size_to) * pad_vocab_size_to
```

Vocab thật là $16\,000 + 9 = 16\,009$ (9 token đặc biệt của nanochat), được **đệm** lên $16\,064$ — bội số 64 gần
nhất. Lý do: nhân ma trận trên tensor core của GPU chạy nhanh hơn hẳn khi chiều ma trận chia hết cho 8 hoặc 64,
và chiều vocab xuất hiện trong ma trận lớn nhất của model (`lm_head`). 55 hàng thừa **không bao giờ được dùng**:
`forward` cắt chúng khỏi logits trước khi tính loss (`logits = logits[..., :self.config.vocab_size]`), nên softmax
chỉ trải trên 16.009 token và các hàng thừa không nhận gradient nào từ loss.

Chi phí của việc đệm nhỏ nhưng đo được, và nó là chi phí *bộ nhớ và tham số*, không phải chi phí chính xác.

In [ ]:
for depth in (6, 8, 10):
    khong_dem = dem_tay(depth, VOCAB, pad_to=1)
    co_dem = dem_tay(depth, VOCAB, pad_to=64)
    thua = co_dem["tong"] - khong_dem["tong"]
    print(f"d{depth}: đệm {VOCAB} -> {co_dem['v_pad']} tốn thêm {thua:,} tham số"
          f" ({thua / co_dem['tong']:.3%} tổng số)")

## 4. FLOPs: chi phí tính toán của một token

**FLOP** (floating-point operation) là một phép cộng hoặc một phép nhân số thực. Quy tắc đếm chuẩn:

- Mỗi tham số tham gia matmul tốn **2 FLOPs** cho mỗi token ở lượt **forward** (một nhân, một cộng dồn).
- Lượt **backward** tốn gấp đôi forward, vì phải tính hai gradient: theo đầu vào và theo trọng số.

Cộng lại: $2 + 4 = 6$ FLOPs mỗi tham số mỗi token, tức

$$C_{\text{token}} \approx 6N_{\text{matmul}}$$

**Ký hiệu mới**
- $C_{\text{token}}$ — FLOPs cho một token trong một bước huấn luyện (forward + backward)
- $N_{\text{matmul}}$ — số tham số có nhân ma trận: phần thân + `lm_head`, không tính các bảng tra cứu

Riêng attention có phần không tỷ lệ với số tham số, vì $QK^\top$ và $AV$ phụ thuộc độ dài ngữ cảnh $T$:

$$C_{\text{attn}} = 12\,h\,d_h\,T_{\text{hiệu dụng}} \quad \text{mỗi lớp}$$

**Vì sao là 12:** hai phép nhân ($QK^\top$ và $AV$) × 2 FLOPs mỗi phép cộng-nhân × 3 (backward gấp đôi forward).

($T_{\text{hiệu dụng}} = \min(T, \text{cửa sổ})$ khi dùng sliding window; project đặt `--window-pattern L` nên
$T_{\text{hiệu dụng}} = T$ ở mọi lớp.) Đây đúng là công thức `GPT.estimate_flops` trong nanochat.

Chú ý: `wte` và `value_embeds` là **tra cứu bảng**, không có matmul, nên không tính vào $N_{\text{matmul}}$ —
chúng tốn bộ nhớ chứ không tốn FLOPs. `lm_head` thì có.

In [ ]:
for depth, seq in ((6, 1024), (8, 1024), (10, 1024)):
    m = dung_model(depth, seq_len=seq)
    n_matmul = m.num_matmul_params()
    f_token = m.estimate_flops()
    than = dem_tay(depth)["than"]
    d, h, dh = 64 * depth, depth * 64 // 128, 128
    f_attn = 12 * h * dh * seq * depth
    print(f"d{depth} (T={seq}):")
    print(f"  tham số matmul        = {n_matmul:,}   (thân {than:,} + lm_head)")
    print(f"  6 × N_matmul          = {6 * n_matmul:,} FLOPs/token")
    print(f"  phần attention        = {f_attn:,} FLOPs/token  ({f_attn / f_token:.1%} tổng)")
    print(f"  nanochat estimate_flops = {f_token:,} FLOPs/token\n")

Ở độ dài ngữ cảnh của project ($T = 1024$) phần attention chiếm khoảng 18–22% FLOPs mỗi token. Nó tăng **tuyến
tính theo $T$** trong khi phần matmul không đổi, nên ở ngữ cảnh dài nó áp đảo. Cell dưới tìm điểm hoà vốn.

In [ ]:
m = dung_model(8, seq_len=65536)
n_matmul = m.num_matmul_params()
print("d8, tỷ trọng attention theo độ dài ngữ cảnh:")
for T in (256, 1024, 4096, 16384, 65536):
    f_attn = 12 * 4 * 128 * T * 8
    print(f"  T = {T:6,}: attention chiếm {f_attn / (6 * n_matmul + f_attn):5.1%} FLOPs mỗi token")

## 5. Từ FLOPs sang thời gian thật: MFU

Nhân FLOPs mỗi token với số token mỗi giây là ra tốc độ tính toán thực tế. Chia cho tốc độ đỉnh của GPU ra
**MFU** (model FLOPs utilization) — tỷ lệ năng lực GPU thực sự biến thành phép tính hữu ích:

$$\mathrm{MFU} = \frac{C_{\text{token}} \times \text{token/giây}}{\text{FLOPS đỉnh của GPU}}$$

**Lưu ý:** **FLOPS** (S hoa) là *tốc độ* — phép tính mỗi giây; **FLOPs** (s thường) là *số lượng* phép tính.

Tesla T4 có đỉnh $65{,}13$ TFLOPS ở fp16 (tensor core). Số token/giây đo được nằm trong
`results/throughput_d*.json` — do cell Gate-0 của notebook 02 ghi ra.

Để đặt con số vào bối cảnh: trên GPU trung tâm dữ liệu với mã tối ưu (FlashAttention, bf16), MFU thường nằm
quanh 30–50%. Project chạy trên T4 với attention SDPA dự phòng và model nhỏ (nhiều phép toán không phải matmul
so với phần matmul), nên kỳ vọng thấp hơn; dưới 10% mới là dấu hiệu GPU chủ yếu đang chờ. Đây là chỉ số nên có
trong báo cáo khi nói về chi phí.

In [ ]:
T4_PEAK = 65.13e12

def doc_throughput():
    ra = {}
    for f in sorted(RUNS.glob("results-v*/results/throughput_d*.json")):
        for ten, v in json.loads(f.read_text()).items():
            ra.setdefault(ten, v)
    return ra


tp = doc_throughput()
print(" run                 | seq_len | tok/s  | FLOPs/token | TFLOPS đạt | MFU   | giờ/lượt")
for ten, v in sorted(tp.items()):
    depth = int(ten.split("_d")[1].split("_")[0])
    seq = seq_len(cpt, ten.split("_d")[0])
    m = dung_model(depth, seq_len=seq)
    f_token = m.estimate_flops()
    dat = f_token * v["tok_per_sec"]
    print(f" {ten:19s} | {seq:7d} | {v['tok_per_sec']:6,} | {f_token:11,} |"
          f" {dat / 1e12:10.2f} | {dat / T4_PEAK:5.1%} | {v['full_run_hours']:7.2f}")

Tổng chi phí tính toán của cả project, theo công thức $C \approx C_{\text{token}} \times D$ với $D$ là số token
đã train:

In [ ]:
tong_flops, tong_gio = 0.0, 0.0
for ten, v in tp.items():
    depth = int(ten.split("_d")[1].split("_")[0])
    seq = seq_len(cpt, ten.split("_d")[0])
    m = dung_model(depth, seq_len=seq)
    D = v["tok_per_sec"] * v["full_run_hours"] * 3600
    tong_flops += m.estimate_flops() * D
    tong_gio += v["full_run_hours"]

print(f"số lượt train có số liệu throughput: {len(tp)}")
print(f"tổng giờ GPU (chỉ các lượt này)    : {tong_gio:.1f} h")
print(f"tổng FLOPs                         : {tong_flops:.3e}")
print(f"                                   = {tong_flops / 1e18:.2f} exaFLOPs")
print(f"\nđể so: GPT-3 175B tốn ~3.1e23 FLOPs, tức gấp {3.1e23 / tong_flops:,.0f} lần cả project này.")

## 6. Chinchilla: bao nhiêu dữ liệu cho một model cỡ này?

Kaplan (2020) và Hoffmann (2022, "Chinchilla") nghiên cứu cùng một câu hỏi: với ngân sách tính toán $C$ cố định,
nên chia cho **model to** hay **dữ liệu nhiều**? Chinchilla kết luận hai thứ nên tăng cùng tốc độ, và quy tắc
ngón tay cái là

$$D^{*} \approx 20 N$$

**Ký hiệu mới**
- $N$ — số tham số (quy ước đếm: bảng ngay dưới); $D$ — số token huấn luyện
- $D^{*}$ — lượng dữ liệu **tối ưu** khi ngân sách tính toán cố định
- $C$ (câu dưới) — tổng FLOPs của cả lượt train, $C \approx 6ND$

tức mỗi tham số nên gặp khoảng 20 token. Với $C \approx 6ND$, điều này xác định luôn cả hai vế khi biết $C$.

Chỗ dễ sai nhất ở đây là **định nghĩa $N$**, vì ba nguồn dùng ba quy ước khác nhau:

| Nguồn | $N$ đếm gì | Tỷ lệ mục tiêu |
|---|---|---|
| Kaplan (2020) | chỉ phần thân (bỏ embedding) | không đưa ra tỷ lệ cố định |
| Chinchilla (2022) | **toàn bộ** tham số, kể cả embedding | $D \approx 20N$ |
| nanochat `--target-param-data-ratio` | `transformer_matrices + lm_head` (xem `get_scaling_params` trong `base_train.py`) | mặc định 12 |

Ở model lớn ba quy ước gần như trùng nhau vì embedding chỉ là phần nhỏ. Ở model cỡ project thì không: embedding
chiếm 59–74% tổng tham số (mục 2), nên cùng một ngân sách token cho ra ba tỷ lệ rất khác nhau.

Project **không** dùng tỷ lệ của nanochat để quyết định số bước, mà đặt thẳng ngân sách token trong
`vitok/conditions.py`:

```python
BUDGET_TOKENS = {6: 250_000_000, 8: 500_000_000, 10: 1_000_000_000}
```

Cell dưới tính tỷ lệ $D/N$ theo cả ba quy ước.

In [ ]:
BUDGET = {6: 250_000_000, 8: 500_000_000, 10: 1_000_000_000}
print(" model | D (token)     | D/N thân (Kaplan) | D/N thân+lm_head (nanochat) | D/N tổng (Chinchilla)")
for depth, D in BUDGET.items():
    t = dem_tay(depth)
    n_nanochat = t["than"] + t["lm_head"]
    print(f" d{depth:<4} | {D:>13,} | {D / t['than']:17.1f} | {D / n_nanochat:27.1f} | {D / t['tong']:21.1f}")
print("\nĐọc đúng: theo quy ước của Chinchilla (toàn bộ tham số), project train với ~6-8 token mỗi tham số,")
print("tức ÍT dữ liệu hơn mức compute-optimal ~20. Theo quy ước của nanochat thì ~15, trên mặc định 12 của nó.")
print("Câu 'D/N ≈ 20 nên đúng chuẩn Chinchilla' chỉ đúng nếu lén đổi sang định nghĩa N của Kaplan — không được viết.")

## 7. Ba cách "cho hai model học bằng nhau"

Đây là mục quan trọng nhất của notebook, vì nó là chỗ một thí nghiệm so tokenizer dễ sai nhất.

Vấn đề: hai tokenizer nén văn bản khác nhau. Với cùng một tập văn bản, SuperBPE sinh ra ít token hơn BPE khoảng
18%. Vậy khi nói "train hai model như nhau" thì **giữ cái gì cố định**?

| Cách | Giữ cố định | Hệ quả |
|---|---|---|
| **Equal-token** | số token | SuperBPE được đọc **nhiều văn bản hơn** → lợi thế không liên quan đến chất lượng tokenizer |
| **Equal-text** | số ký tự văn bản | hai bên đọc cùng lượng nội dung; số token và FLOPs khác nhau |
| **Equal-FLOPs** | tổng phép tính | hai bên tốn cùng compute; lượng văn bản khác nhau |

Project chọn **equal-text**, vì câu hỏi nghiên cứu là "với cùng lượng văn bản tiếng Việt, tokenizer nào cho model
học tốt hơn". Cách cài đặt, trong `vitok/conditions.py`:

$$T_{\text{cond}} = \mathrm{round}_8\!\left(T_{\text{base}} \cdot \frac{\text{chars/token của bpe-nfc}}{\text{chars/token của cond}}\right)$$

**Ký hiệu mới:** $T_{\text{cond}}$ — `max_seq_len` của điều kiện đang xét; $\mathrm{round}_8$ — làm tròn tới
bội số gần nhất của 8.

với $T_{\text{base}} = 1024$. Số bước và số chuỗi mỗi bước (64) giữ nguyên cho mọi điều kiện, nên **số ký tự mỗi
bước là bằng nhau** trong khi số token mỗi bước khác nhau.

In [ ]:
print(" điều kiện  | chars/token | seq_len | token/bước | ký tự/bước (xấp xỉ)")
for cond in ("bpe-nfc", "bpe-nfd", "super-nfc", "super-nfd"):
    T = seq_len(cpt, cond)
    tok_buoc = SEQS_PER_STEP * T
    print(f" {cond:10s} | {cpt[cond]:11.4f} | {T:7d} | {tok_buoc:10,} | {tok_buoc * cpt[cond]:14,.0f}")

print(f"\nchênh lệch ký tự/bước giữa điều kiện tốn nhất và ít nhất: "
      f"{max(SEQS_PER_STEP * seq_len(cpt, c) * cpt[c] for c in cpt) / min(SEQS_PER_STEP * seq_len(cpt, c) * cpt[c] for c in cpt) - 1:.2%}")
print("(không bằng 0 tuyệt đối vì seq_len phải làm tròn về bội số 8)")

Equal-text **không** đồng nghĩa equal-compute: điều kiện nào ít token hơn thì tốn ít FLOPs hơn cho cùng lượng văn
bản. Đây là một hệ quả bắt buộc phải ghi vào phần hạn chế của báo cáo, kèm hướng của nó.

In [ ]:
print(" điều kiện  | seq_len | FLOPs/token | FLOPs mỗi bước | so với bpe-nfc | giờ đo được (d8)")
goc = None
for cond in ("bpe-nfc", "bpe-nfd", "super-nfc", "super-nfd"):
    T = seq_len(cpt, cond)
    m = dung_model(8, seq_len=T)
    f_buoc = m.estimate_flops() * SEQS_PER_STEP * T
    goc = goc or f_buoc
    gio = tp.get(f"{cond}_d8_s0", {}).get("full_run_hours", float("nan"))
    print(f" {cond:10s} | {T:7d} | {m.estimate_flops():11,} | {f_buoc:14,} | {f_buoc / goc:13.3f} | {gio:16.2f}")

print("\nSuperBPE nhận ÍT compute hơn cho cùng lượng văn bản (~20%).")
print("Hướng của thiên lệch: bất lợi cho SuperBPE -> kết quả 'SuperBPE hơi kém hơn ở d8/d10'")
print("có thể một phần do compute, và đây là hạn chế phải ghi rõ, không được bỏ qua.")

## 8. Ghim learning rate: vì sao cần `--scaling-batch-size`

nanochat chỉnh learning rate theo kích thước batch, vì batch lớn cho gradient ít nhiễu hơn nên chịu được bước đi
dài hơn. Quy tắc dùng cho AdamW là

$$\eta \propto \sqrt{B / B_{\text{ref}}}, \qquad B_{\text{ref}} = 2^{19} = 524\,288 \text{ token}$$

**Ký hiệu mới**
- $\eta$ — learning rate; $\propto$ — "tỷ lệ thuận với"
- $B$ — batch mỗi bước, tính bằng **token** (khác $B$ = số byte ở notebook 01)

và weight decay theo

$$\lambda = \lambda_{\text{ref}} \sqrt{B/B_{\text{ref}}} \cdot \frac{D_{\text{ref}}}{D}$$

**Ký hiệu mới:** $\lambda$ — weight decay; $D$, $D_{\text{ref}}$ — tổng số token của lượt này và của lượt tham
chiếu d12 (`D_REF` trong `base_train.py`).

Vấn đề với thiết kế equal-text: `total_batch_size` đo bằng **token**, mà số token mỗi bước lại khác nhau giữa các
điều kiện. Nếu để nguyên, SuperBPE sẽ tự động nhận learning rate thấp hơn BPE — một khác biệt **do tokenizer gây
ra gián tiếp qua bộ lập lịch**, không phải do bản chất tokenizer, và nó sẽ lẫn vào kết quả.

Bản vá `patches/nanochat.patch` thêm `--scaling-batch-size` để mọi điều kiện dùng chung một batch tham chiếu
($64 \times 1024$ token của bpe-nfc) khi tính hệ số này. Cell dưới đo đúng phần chênh lệch đã bị loại bỏ.

In [ ]:
B_REF = 2 ** 19
base_batch = SEQS_PER_STEP * BASE_SEQ
print(" điều kiện  | batch thật (token) | η scale nếu KHÔNG ghim | η scale khi ghim")
for cond in ("bpe-nfc", "super-nfc", "super-nfd"):
    T = seq_len(cpt, cond)
    batch = SEQS_PER_STEP * T
    print(f" {cond:10s} | {batch:18,} | {math.sqrt(batch / B_REF):22.4f} | {math.sqrt(base_batch / B_REF):16.4f}")

lech = 1 - math.sqrt(SEQS_PER_STEP * seq_len(cpt, "super-nfc") / base_batch)
print(f"\nkhông ghim: SuperBPE sẽ nhận learning rate thấp hơn BPE {lech:.1%}")
print("=> một confound do lập lịch, không phải do tokenizer. Bản vá loại bỏ đúng phần này.")

cfg = train_args(cpt, "super-nfc", 8)
print("\ntham số dòng lệnh sinh ra cho super-nfc d8:")
for k, v in cfg.items():
    print(f"  --{k}={v}")

## 9. Scaling law: khớp đường cong qua ba điểm

Dạng hàm chuẩn của scaling law theo số tham số, với dữ liệu đủ nhiều:

$$L(N) = \frac{A}{N^{\alpha}} + L_{\infty}$$

**Lưu ý:** $\alpha$ ở đây là số mũ của scaling law, không phải mức ý nghĩa ở notebook 02.

Ba thành phần: $L_\infty$ là **entropy không thể giảm** của ngôn ngữ (phần không model nào loại bỏ được, xem
notebook 01), $A/N^\alpha$ là phần sai lệch giảm dần theo cỡ model, và $\alpha$ cho biết giảm nhanh cỡ nào.
Giá trị của $\alpha$ phụ thuộc dạng hàm: Kaplan khớp dạng **không có** $L_\infty$ và được $\alpha \approx 0{,}076$;
Chinchilla khớp dạng **có** $L_\infty$ (cùng một số hạng theo $D$) và được $\alpha \approx 0{,}34$. Hai con số
không so với nhau được vì chúng là tham số của hai hàm khác nhau.

Project có đúng **ba** điểm (d6, d8, d10), mà hàm có ba tham số, nên đây là nội suy khít chứ không phải khớp có
dư bậc tự do: không kiểm chứng được dạng hàm, chỉ dùng để nội suy trong khoảng đã đo và để thấy độ cong. Mọi ngoại
suy ra ngoài phải nói rõ là suy đoán.

Cách khớp không cần scipy: quét $\alpha$ trên lưới; với mỗi $\alpha$ cố định, $L$ là hàm **tuyến tính** theo
$x = N^{-\alpha}$, nên $A$ và $L_\infty$ giải được bằng bình phương tối thiểu tuyến tính (`np.linalg.lstsq`).

In [ ]:
N = np.array([dem_tay(d)["than"] for d in (6, 8, 10)], float)
bpc_bpe = np.array([1.0914, 1.0019, 0.9370])      # bảng d6/d8/d10 trong results/summary.md
bpc_super = np.array([1.0896, 1.0037, 0.9399])


def khop(N, y):
    tot = None
    for alpha in np.linspace(0.01, 0.60, 6000):
        X = np.stack([N ** (-alpha), np.ones_like(N)], axis=1)
        (A, Linf), *_ = np.linalg.lstsq(X, y, rcond=None)
        sai_so = float(((X @ [A, Linf] - y) ** 2).sum())
        if tot is None or sai_so < tot[0]:
            tot = (sai_so, alpha, A, Linf)
    return tot


sai_so, alpha, A, Linf = khop(N, bpc_bpe)
print(f"khớp bpe-nfc: α = {alpha:.4f}, A = {A:.4g}, L∞ = {Linf:.4f} bpc  (sai số bình phương {sai_so:.2e})")
print("\n model | N thân     | bpc đo  | bpc khớp")
for d, n, y in zip((6, 8, 10), N, bpc_bpe):
    print(f" d{d:<4} | {n:>10,.0f} | {y:.4f}  | {A * n ** -alpha + Linf:.4f}")

for d in (12, 16, 20):
    n = 12 * (64 * d) ** 2 * d
    print(f" d{d} (ngoại suy, KHÔNG kiểm chứng được): N = {n:,} -> bpc ≈ {A * n ** -alpha + Linf:.4f}")

Kết quả khớp cho $L_\infty$ **âm** ($\approx -0{,}14$ bpc). Entropy không thể âm, nên con số này tự nó báo rằng
phép khớp đang dùng $L_\infty$ như một tham số hình học để uốn đường cong qua ba điểm, chứ không đo được đại
lượng mà ký hiệu đó đặt tên.

$L_\infty$ khớp được ở đây **không** phải entropy thật của tiếng Việt: nó chỉ là tham số làm đường cong đi qua ba
điểm, và với ba điểm thì nó cực kỳ không ổn định. Cách kiểm tra nhanh mức không ổn định đó: xê dịch một điểm đo
đúng bằng độ lớn nhiễu seed ($0{,}0005$ bpc, notebook 02 mục 11) rồi khớp lại.

In [ ]:
print("độ nhạy của các tham số khớp khi xê dịch từng điểm ±0,0005 bpc (đúng bằng nhiễu seed):")
print("  điểm bị xê dịch | α      | L∞")
print(f"  (không xê dịch) | {alpha:.4f} | {Linf:.4f}")
for i, ten in enumerate(("d6", "d8", "d10")):
    for dau in (+1, -1):
        y = bpc_bpe.copy()
        y[i] += dau * 0.0005
        _, a2, _, l2 = khop(N, y)
        print(f"  {ten} {dau:+d}·0,0005  | {a2:.4f} | {l2:.4f}")
print("\nL∞ nhảy từ khoảng -0,42 tới +0,05 (đổi cả dấu) chỉ vì một thay đổi nằm trong nhiễu")
print("-> không báo cáo L∞ như một kết quả. α nhỏ (~0,09) cũng là dấu hiệu: với L∞ âm, phép khớp đang")
print("   suy biến về gần dạng luỹ thừa thuần kiểu Kaplan, chứ không xác định được đường cong có tiệm cận.")

## 10. H3: hiệu ứng thay đổi thế nào theo cỡ model

Giả thuyết H3 của project: lợi ích của SuperBPE **phụ thuộc cỡ model**. Số đo được:

| $\Delta$ = super-nfc − bpe-nfc | d6 | d8 | d10 |
|---|---|---|---|
| bpc clean | $-0{,}0018$ | $+0{,}0018$ | $+0{,}0028$ |

Dấu đổi từ âm sang dương và độ lớn tăng đơn điệu theo $N$. Vì trục $N$ trải trên chưa tới một bậc độ lớn
($10{,}6\text{M} \to 49{,}2\text{M}$, tức 4,6 lần), cách trình bày an toàn là hồi quy tuyến tính theo $\log N$ và
nói rõ đây là mô tả xu hướng trong khoảng đã đo, không phải quy luật.

Điểm giao (nơi $\Delta = 0$) là con số dễ bị lạm dụng nhất: nó nằm **giữa** hai điểm đo nên nội suy được, nhưng
mọi phát biểu kiểu "SuperBPE có lợi cho model dưới X tham số" cần kèm khoảng bất định, mà với ba điểm thì khoảng
đó rất rộng.

In [ ]:
delta = bpc_super - bpc_bpe
x = np.log10(N)
he_so = np.polyfit(x, delta, 1)
print(" model | N thân     | Δ đo được | Δ theo đường thẳng")
for d, n, dd in zip((6, 8, 10), N, delta):
    print(f" d{d:<4} | {n:>10,.0f} | {dd:+.4f}   | {np.polyval(he_so, math.log10(n)):+.4f}")

giao = 10 ** (-he_so[1] / he_so[0])
print(f"\nhệ số góc = {he_so[0]:+.5f} bpc trên mỗi bậc 10 của N")
print(f"điểm giao Δ = 0 tại N ≈ {giao:,.0f} tham số thân")
print(f"  -> nằm giữa d6 ({N[0]:,.0f}) và d8 ({N[1]:,.0f}): nội suy, không phải ngoại suy")

for d in (12, 16):
    n = 12 * (64 * d) ** 2 * d
    print(f"  ngoại suy d{d}: Δ ≈ {np.polyval(he_so, math.log10(n)):+.4f} bpc  (suy đoán)")

print(f"\nthanh nhiễu seed ±0,0005 so với độ lớn hiệu ứng lớn nhất {abs(delta).max():.4f}:"
      f" gấp {abs(delta).max() / 0.0005:.1f} lần -> xu hướng đọc được, nhưng chỉ với 3 điểm.")

## 11. Tóm tắt

| Đại lượng | Công thức | Số của project |
|---|---|---|
| Chiều residual | $d = 64L$ | 384 / 512 / 640 |
| Tham số phần thân | $N = 12d^2L$ | 10,6M / 25,2M / 49,2M |
| Tổng tham số | $N + 2V_{\text{pad}}d + n_{ve}V_{\text{pad}}d$ | 41,5M / 74,5M / 121,1M |
| FLOPs mỗi token | $6N_{\text{matmul}} + 12hd_hT$ mỗi lớp | xem mục 4 |
| Tổng compute | $C \approx C_{\text{token}} \cdot D$ | ~1,6 exaFLOPs cho 14 lượt train |
| MFU | $C_{\text{token}} \cdot \text{tok/s} / \text{FLOPS đỉnh}$ | xem mục 5 |
| Chinchilla | $D^{*} \approx 20N$, $N$ = toàn bộ tham số | project ~6–8, tức ít dữ liệu hơn mức tối ưu |
| Equal-text | $T_{\text{cond}} = T_{\text{base}} \cdot \mathrm{cpt}_{\text{base}}/\mathrm{cpt}_{\text{cond}}$ | 1024 → 840 |
| Ghim LR | $\eta \propto \sqrt{B_{\text{ghim}}/B_{\text{ref}}}$ | loại bỏ chênh lệch 9,4% |
| Scaling law | $L(N) = A N^{-\alpha} + L_\infty$ | 3 điểm, chỉ nội suy |

## 12. Câu hỏi tự kiểm

1. Vì sao phần thân tăng $\approx 4{,}6$ lần khi đi từ d6 lên d10, trong khi số lớp chỉ tăng 1,67 lần?
2. Model d6 có 41,5M tham số nhưng chỉ 10,6M ở phần thân. Con số nào nên đặt vào bảng scaling, và vì sao?
3. Vì sao `wte` và `value_embeds` không tính vào FLOPs còn `lm_head` thì có?
4. Ở độ dài ngữ cảnh nào thì attention chiếm quá nửa FLOPs mỗi token của d8?
5. Nếu chuyển từ equal-text sang equal-token, điều kiện nào được lợi, và lợi ở đâu?
6. Không có `--scaling-batch-size` thì learning rate của super-nfc lệch bao nhiêu phần trăm so với bpe-nfc, và
   vì sao đó là confound chứ không phải một phần của hiệu ứng tokenizer?
7. Vì sao không nên báo cáo $L_\infty$ khớp được ở mục 9?
8. Điểm giao $\Delta = 0$ nằm giữa d6 và d8. Phát biểu nào về nó là hợp lệ, phát biểu nào là ngoại suy quá tay?

**Đáp án gợi ý**

1. Vì $d = 64L$ nên $N = 12d^2L \propto L^3$; $(10/6)^3 = 4{,}63$.
2. Đặt **cả hai** và nói rõ quy ước. Phần thân là trục ổn định hơn để so giữa các tokenizer khác vocab (tỷ lệ bảng
   tra cứu đổi theo vocab), và là quy ước của Kaplan; nhưng mọi so sánh với Chinchilla phải dùng tổng tham số.
3. Vì embedding là tra cứu hàng (không nhân ma trận), còn `lm_head` nhân vector residual với ma trận
   $d \times V_{\text{pad}}$ ở mọi vị trí.
4. Chạy cell mục 4: quanh $T \approx 4096$ với d8 (attention đạt 50,1%).
5. SuperBPE, vì cùng số token nghĩa là nhiều văn bản hơn khoảng 18% — lợi thế về dữ liệu chứ không phải về chất
   lượng tokenizer.
6. Thấp hơn 9,4%; đó là hệ quả của bộ lập lịch phản ứng với số token mỗi bước, hoàn toàn tách rời khỏi câu hỏi
   "tokenizer nào giúp model học tốt hơn".
7. Vì với ba điểm và ba tham số, $L_\infty$ không có bậc tự do dư để kiểm chứng, và xê dịch một điểm trong phạm vi
   nhiễu seed đã làm nó nhảy rất mạnh (cell cuối mục 9).
8. Hợp lệ: "trong khoảng 10,6M–49,2M tham số, hiệu ứng đổi dấu quanh khoảng $2 \times 10^{7}$". Quá tay: "SuperBPE
   luôn có hại với model trên X tham số" — ba điểm không chống đỡ nổi phát biểu đó.

**Nguồn đọc thêm**

- [Scaling Laws for Neural Language Models](https://arxiv.org/abs/2001.08361) — mục 2–3, dạng hàm và cách khớp.
- [Training Compute-Optimal Large Language Models](https://arxiv.org/abs/2203.15556) — quy tắc $D \approx 20N$.
- [The FLOPs calculus of language model training](https://medium.com/@dzmitrybahdanau/the-flops-calculus-of-language-model-training-3b19c1f025e4)
  — nguồn của quy tắc 6 FLOPs mỗi tham số, chính là link trong `nanochat/gpt.py`.
- [SuperBPE](https://arxiv.org/abs/2503.13423) — mục thiết lập thí nghiệm: họ cố định ngữ cảnh theo byte và
  chỉnh số bước để **bằng FLOPs**, khác thiết kế equal-text của project (mục 7 ở đây).